# Resampling con Bootstrap

|                |   |
:----------------|---|
| **Nombre**     |  Jorge Oviedo Magaña |
| **Fecha**      | 27/04/2026 |
| **Expediente** |  757048 | 

In [3]:
import pandas as pd

df = pd.read_excel("Motor Trend Car Road Tests.xlsx")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 12 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   model   32 non-null     object 
 1   mpg     32 non-null     float64
 2   cyl     32 non-null     int64  
 3   disp    32 non-null     float64
 4   hp      32 non-null     int64  
 5   drat    32 non-null     float64
 6   wt      32 non-null     float64
 7   qsec    32 non-null     float64
 8   vs      32 non-null     int64  
 9   am      32 non-null     int64  
 10  gear    32 non-null     int64  
 11  carb    32 non-null     int64  
dtypes: float64(5), int64(6), object(1)
memory usage: 3.1+ KB


## Regresión lineal

In [4]:
import statsmodels.api as sm

y = df[["mpg"]]
X = df[["hp","qsec"]]
X = sm.add_constant(X)  # agrega β₀

# Paso 2 - Regresión lineal
modelo = sm.OLS(y, X).fit()
print(modelo.summary())

# Paso 2a - Intervalos de confianza
print("\nIntervalos de confianza (95%):")
print(modelo.conf_int())

#resampling

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        16:37:21   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

## Bootstrap (1000 regresiones)

In [5]:
import numpy as np

np.random.seed(42)
n_bootstrap = 1000
betas_bootstrap = []

for _ in range(n_bootstrap):
    # Muestra con reemplazo (mismo tamaño que el original)
    muestra = df.sample(n=len(df), replace=True)
    
    y_b = muestra["mpg"]
    X_b = sm.add_constant(muestra[["hp", "qsec"]])
    
    modelo_b = sm.OLS(y_b, X_b).fit()
    betas_bootstrap.append(modelo_b.params)

betas_bootstrap = pd.DataFrame(betas_bootstrap)
print(betas_bootstrap.describe())

             const           hp         qsec
count  1000.000000  1000.000000  1000.000000
mean     50.203330    -0.088136    -0.969126
std      11.136038     0.016787     0.533181
min      18.807867    -0.144922    -3.370591
25%      42.883804    -0.099042    -1.273952
50%      49.214812    -0.087085    -0.914907
75%      56.547040    -0.076195    -0.594599
max     101.883926    -0.040363     0.782246


## Comparación

In [6]:
# Intervalos de confianza bootstrap
ic_bootstrap = pd.DataFrame({
    "lower": betas_bootstrap.quantile(0.025),
    "upper": betas_bootstrap.quantile(0.975)
})

# Intervalos de confianza OLS (paso 2)
ic_ols = modelo.conf_int()
ic_ols.columns = ["lower", "upper"]

print("=== Paso 2 - OLS  ===")
print(ic_ols)
print("\n=== Paso 3 - Bootstrap  ===")
print(ic_bootstrap)

=== Paso 2 - OLS  ===
           lower      upper
const  25.614894  71.032516
hp     -0.113089  -0.056097
qsec   -1.979929   0.206770

=== Paso 3 - Bootstrap  ===
           lower      upper
const  31.722706  75.857555
hp     -0.123376  -0.058694
qsec   -2.200632  -0.090795


## Conclusión

En general ambos pasos tienen intervalos de confianza muy similares lo que significa que el resampling fue exitoso  y se puede usar en otros contextos.

# Agregating

Realizar mil modelos y tomar tres columnas al azar (31x10)
* cada modelo (16x3)
* Predict, test, y_hat test, R2 test

In [7]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# Columnas disponibles (sin mpg ni model)
features = ["cyl", "disp", "hp", "drat", "wt", "qsec", "vs", "am", "gear", "carb"]

y = df["mpg"]
X = df[features]

# Split 50/50
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

modelos = []

### Realizo los 1000 modelos tomando 3 columnas al azar (train)

In [8]:
for i in range(1000):
    # 3 columnas al azar
    vars_i = np.random.choice(features, size=3, replace=False)
    
    X_train_i = X_train[vars_i]
    X_test_i  = X_test[vars_i]
    
    # Entrenar
    model_i = LinearRegression()
    model_i.fit(X_train_i, y_train)

    modelos.append({
        "vars": list(vars_i),
        "model": model_i
    })

### Ahora con Test

In [9]:
# LOOP TEST
resultados = []

for i in range(1000):
    vars_i  = modelos[i]["vars"]
    model_i = modelos[i]["model"]
    
    X_test_i = X_test[vars_i]
    
    y_pred_i = model_i.predict(X_test_i)
    
    resultados.append({
        "vars"  : vars_i,
        "y_pred": y_pred_i
    })

### Resultados

In [10]:
import numpy as np

# Matriz de 1000 x 16
y_preds_matrix = np.array([r["y_pred"] for r in resultados])

# Promedio por columna (por observación)
y_pred_final = y_preds_matrix.mean(axis=0)

print(y_pred_final)

[20.36651583 10.25189512 14.39866191 27.1758539  23.56844571 20.16553001
 13.70801415 27.49342317 15.34211211 21.81299247 15.42127129 10.49931102
 19.76392628 15.3162746  14.82077627 13.51611226]


### Compararlo con la y_test original

In [11]:
# Comparar y_pred_final vs y_test
comparacion = pd.DataFrame({
    "y_test"      : y_test.values,
    "y_pred_final": y_pred_final
})

print(comparacion)

    y_test  y_pred_final
0     19.7     20.366516
1     10.4     10.251895
2     19.2     14.398662
3     32.4     27.175854
4     22.8     23.568446
5     19.2     20.165530
6     15.0     13.708014
7     27.3     27.493423
8     17.3     15.342112
9     21.0     21.812992
10    18.7     15.421271
11    14.7     10.499311
12    18.1     19.763926
13    15.2     15.316275
14    16.4     14.820776
15    13.3     13.516112


In [12]:
# R² final
from sklearn.metrics import r2_score

r2_final = r2_score(y_test, y_pred_final)
print(f"R² final (promedio de 1000 modelos): {r2_final:.4f}")

R² final (promedio de 1000 modelos): 0.7868


# Random Forest

### Sin Optimización

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest base
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [18]:
# Predecir y evaluar
y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)
print(f"R² test (Random Forest base): {r2_rf:.4f}")

R² test (Random Forest base): 0.8376


### Optimizando los hiperparametros (# trees, max depth, max leaves)

In [51]:
from sklearn.model_selection import GridSearchCV, KFold

param_grid = {
    "n_estimators"  : [500, 700, 1000, 1500],
    "max_depth"     : [None, 5, 10, 15],
    "max_leaf_nodes": [None, 20, 50, 100]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [57]:
grid_search = GridSearchCV(
    estimator     = RandomForestRegressor(random_state=42),
    param_grid    = param_grid,
    scoring       = "r2",
    cv            = kf,
    n_jobs        = -1,
    verbose       = 1,
    refit         = True
)

grid_search.fit(X, y)

Fitting 5 folds for each of 64 candidates, totalling 320 fits


,estimator,RandomForestR...ndom_state=42)
,param_grid,"{'max_depth': [None, 5, ...], 'max_leaf_nodes': [None, 20, ...], 'n_estimators': [500, 700, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,1500


In [58]:
mejor_rf = grid_search.best_estimator_
y_pred = mejor_rf.predict(X)
r2 = r2_score(y, y_pred)

print("Mejores hiperparámetros:")
print(grid_search.best_params_)
print(f"\nR² (modelo completo): {r2:.4f}")

Mejores hiperparámetros:
{'max_depth': None, 'max_leaf_nodes': None, 'n_estimators': 1500}

R² (modelo completo): 0.9776


# Boosting

In [59]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(random_state=42, max_depth=None, max_leaf_nodes= None, n_estimators= 1500)
gb.fit(X, y)

y_pred_gb = gb.predict(X)
r2_gb = r2_score(y, y_pred_gb)
print(f"R² Gradient Boosting base: {r2_gb:.4f}")

R² Gradient Boosting base: 1.0000
